In [ ]:
import json
import gzip
import os
import csv
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer

# Define global pipeline paths
TARGET_FILE = "candidates.jsonl"
OUTPUT_FILE = "submission.csv"

In [ ]:
def stream_and_filter_candidates(compressed_file_path):
    """
    Streams candidates from the massive gzipped JSONL dataset line-by-line,
    filtering out Honeypots and IT Outsourcing service giants on the fly.
    """
    service_giants = {"tcs", "infosys", "wipro", "cognizant", "accenture", "tech mahindra", "mindtree"}
    passed_candidates = []
    honeypot_count = 0
    service_giant_count = 0
    total_processed = 0

    print(f"🚀 Open and streaming file data pipeline: {compressed_file_path}...")

    # Change from gzip.open to open as the file is not gzipped
    with open(compressed_file_path, 'r', encoding='utf-8') as file:
        for line in file:
            total_processed += 1
            candidate = json.loads(line.strip())

            profile = candidate.get("profile", {})
            career_history = candidate.get("career_history", [])
            skills = candidate.get("skills", [])

            # --- TRAP 1 CHECK: Honeypot Detection ---
            years_exp = profile.get("years_of_experience", 0)
            total_allowed_months = years_exp * 12
            is_honeypot = False

            for skill in skills:
                if skill.get("duration_months", 0) > (total_allowed_months + 2):
                    is_honeypot = True
                    break

            if is_honeypot:
                honeypot_count += 1
                continue

            # --- TRAP 2 CHECK: IT Service Giant Filtering ---
            all_companies = [str(job.get("company", "")).lower() for job in career_history]
            if all_companies and all(any(giant in comp for giant in all_companies) for comp in service_giants):
                service_giant_count += 1
                continue

            passed_candidates.append(candidate)

    print("\n🏁 --- Streaming Phase Complete ---")
    print(f"Total Profiles Streamed: {total_processed}")
    print(f"❌ Honeypot Traps Defused & Removed: {honeypot_count}")
    print(f"❌ IT Service Giant Profiles Excluded: {service_giant_count}")
    print(f"✅ High-Quality Candidates Passed to Pool: {len(passed_candidates)}")

    return passed_candidates

In [ ]:
def calculate_skill_score_final(candidate):
    """
    Calculates a verified technical competency score.
    Ensures case-insensitive lookups for both target skills and platform tests.
    """
    target_skills = {
        "nlp", "embeddings", "vector search", "fine-tuning llms",
        "mlops", "information retrieval", "dense retrieval", "hybrid search",
        "pinecone", "weaviate", "qdrant", "milvus", "faiss", "opensearch", "elasticsearch"
    }

    candidate_skills = candidate.get("skills", [])
    signals = candidate.get("redrob_signals", {})

    raw_assessments = signals.get("skill_assessment_scores", {})
    assessment_scores_lowercase = {k.lower(): v for k, v in raw_assessments.items()}

    total_skill_weight = 0.0
    matched_core_count = 0
    proficiency_map = {"beginner": 1, "intermediate": 2, "advanced": 3, "expert": 4}

    for skill in candidate_skills:
        skill_name_lower = skill.get("name", "").lower()

        if skill_name_lower in target_skills:
            matched_core_count += 1
            prof_str = skill.get("proficiency", "beginner")
            base_multiplier = proficiency_map.get(prof_str, 1)
            duration = skill.get("duration_months", 0)
            endorsements = skill.get("endorsements", 0)

            calculated_trust = (duration * 0.1) + (endorsements * 0.05)
            skill_entry_score = base_multiplier * (1 + calculated_trust)

            if skill_name_lower in assessment_scores_lowercase:
                official_score = assessment_scores_lowercase[skill_name_lower]
                skill_entry_score = (skill_entry_score * 0.4) + (official_score * 0.6)

            total_skill_weight += skill_entry_score

    if matched_core_count >= 3:
        total_skill_weight *= 1.25

    return total_skill_weight

In [ ]:
def calculate_behavioral_multiplier(candidate):
    """
    Computes a multiplicative modifier based on real platform activity metrics.
    """
    signals = candidate.get("redrob_signals", {})
    multiplier = 1.0

    if signals.get("open_to_work_flag", False) is True:
        multiplier *= 1.15

    last_active_str = signals.get("last_active_date", "")
    if last_active_str:
        try:
            last_active = datetime.strptime(last_active_str, "%Y-%m-%d")
            benchmark_date = datetime(2026, 6, 19)
            days_inactive = (benchmark_date - last_active).days

            if days_inactive > 180:
                multiplier *= 0.50
            elif days_inactive > 90:
                multiplier *= 0.75
            elif days_inactive <= 30:
                multiplier *= 1.10
        except ValueError:
            pass

    response_rate = signals.get("recruiter_response_rate", 0.0)
    if response_rate >= 0.85:
        multiplier *= 1.10
    elif response_rate < 0.40:
        multiplier *= 0.70

    completion_rate = signals.get("interview_completion_rate", 0.0)
    if completion_rate >= 0.90:
        multiplier *= 1.05
    elif completion_rate < 0.60:
        multiplier *= 0.80

    github_score = signals.get("github_activity_score", -1)
    if github_score >= 75:
        multiplier *= 1.15
    elif github_score == -1 or github_score < 10:
        multiplier *= 0.95

    return multiplier

def min_max_normalize(vector):
    """Scales numeric arrays strictly between 0.0 and 1.0."""
    arr = np.array(vector, dtype=np.float32)
    min_val = arr.min()
    max_val = arr.max()
    if max_val == min_val:
        return np.zeros_like(arr)
    return (arr - min_val) / (max_val - min_val)

In [ ]:
def run_production_semantic_pipeline(passed_candidates, output_filename="submission.csv"):
    """
    True Semantic Pipeline: Batches text embeddings across ALL candidates,
    normalizes features to prevent scaling dominance, applies behavioral signals,
    and outputs hyper-customized, data-tiered professional justifications.
    """
    print("📦 Loading local semantic embedding model...")
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    jd_anchor_text = (
        "Senior AI Engineer Founding Team. Deep technical depth in modern ML systems, "
        "embeddings, retrieval, ranking, LLMs, and fine-tuning. Scrappy product-engineering "
        "attitude comfortable building production RAG pipelines, search architectures, "
        "vector indexes, and evaluation frameworks from scratch in a fast-paced startup."
    )
    jd_vector = embedding_model.encode(jd_anchor_text, convert_to_numpy=True)

    print(f"⏳ Extracting narratives and computing technical skills for all {len(passed_candidates)} candidates...")
    narratives = []
    raw_skill_scores = []

    for cand in passed_candidates:
        raw_skill_scores.append(calculate_skill_score_final(cand))
        prof = cand.get("profile", {})
        narrative = f"Title: {prof.get('current_title', '')}. Headline: {prof.get('headline', '')}. Summary: {prof.get('summary', '')}"
        narratives.append(narrative)

    print("📦 Vectorizing profiles via Batched Sentence Transformers (MiniLM)...")
    candidate_embeddings = embedding_model.encode(narratives, batch_size=128, show_progress_bar=True, convert_to_numpy=True)

    print("🎯 Calculating structural cosine similarities against Job Description...")
    norms_jd = np.linalg.norm(jd_vector)
    norms_candidates = np.linalg.norm(candidate_embeddings, axis=1)
    norms_candidates[norms_candidates == 0] = 1.0
    raw_semantic_scores = np.dot(candidate_embeddings, jd_vector) / (norms_jd * norms_candidates)

    print("⚖️ Applying Min-Max scaling to equalize score boundaries...")
    normalized_skills = min_max_normalize(raw_skill_scores)
    normalized_semantics = min_max_normalize(raw_semantic_scores)

    print("🔄 Blending indices and computing behavioral platform weights...")
    final_scored_pool = []

    for idx, cand in enumerate(passed_candidates):
        composite_base = (normalized_skills[idx] * 0.4) + (normalized_semantics[idx] * 0.6)
        behavioral_mod = calculate_behavioral_multiplier(cand)
        final_score = round(float(composite_base * behavioral_mod), 4)

        final_scored_pool.append({
            "candidate_id": cand.get("candidate_id"),
            "score": final_score,
            "raw_semantic": round(float(raw_semantic_scores[idx]), 2),
            "profile": cand.get("profile", {}),
            "signals": cand.get("redrob_signals", {})
        })

    # Strict Tie-Breaker Sorting Rule
    final_scored_pool.sort(key=lambda x: (-x["score"], x["candidate_id"]))
    top_100 = final_scored_pool[:100]

    print(f"📝 Writing exactly {len(top_100)} highly calibrated records to '{output_filename}'...")

    with open(output_filename, mode="w", encoding="utf-8", newline="") as csv_file:
        fieldnames = ["candidate_id", "rank", "score", "reasoning"]
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()

        for index, item in enumerate(top_100):
            rank = index + 1
            prof = item["profile"]
            sigs = item["signals"]

            title = prof.get("current_title", "Engineer")
            exp = prof.get("years_of_experience", 0)
            sem_align = item["raw_semantic"]

            # --- DATA-DRIVEN TIERED ADJECTIVE EVALUATIONS ---
            resp_rate = int(sigs.get("recruiter_response_rate", 0.0) * 100)
            if resp_rate >= 85:
                resp_label = "highly responsive recruiter interaction"
            elif resp_rate < 40:
                resp_label = "limited recruiter responsiveness"
            else:
                resp_label = "moderate recruiter engagement"

            completion = int(sigs.get("interview_completion_rate", 0.0) * 100)
            if completion >= 90:
                comp_label = "excellent interview attendance history"
            elif completion < 60:
                comp_label = "unreliable interview completion metrics"
            else:
                comp_label = "stable interview consistency"

            active_str = sigs.get("last_active_date", "")
            try:
                days_ago = (datetime(2026, 6, 19) - datetime.strptime(active_str, "%Y-%m-%d")).days
                # --- BUGFIX: BAKE THE GRAMMAR ARTICLE INTO THE LOGIC BRANCHES ---
                active_context = f"an active footprint (last seen {days_ago}d ago)" if days_ago <= 30 else "a passive candidate footprint"
            except:
                active_context = "a tracked platform activity history"

            # Construct cohesive professional rationale for human review judges
            reasoning = (
                f"{title} with {exp} years of history showing a strong {sem_align} semantic fit for the JD. "
                f"Demonstrates {resp_label} ({resp_rate}%), {comp_label} ({completion}%), and {active_context}."
            )

            writer.writerow({
                "candidate_id": item["candidate_id"],
                "rank": rank,
                "score": item["score"],
                "reasoning": reasoning
            })

    print(f"🏁 SUCCESS! Standalone semantic-first submission exported to '{output_filename}'")

In [ ]:
def deploy_validation_script():
    """Generates the official validate_submission.py file inside the active workspace."""
    validate_code = """
import csv
import re
from pathlib import Path

REQUIRED_HEADER = ["candidate_id", "rank", "score", "reasoning"]
CANDIDATE_ID_PATTERN = re.compile(r"^CAND_[0-9]{7}$")

def validate_submission(csv_path):
    errors = []
    path = Path(csv_path)
    if path.suffix.lower() != ".csv":
        errors.append("Filename must use a .csv extension.")
    try:
        with open(path, "r", encoding="utf-8", newline="") as f:
            reader = csv.reader(f)
            try:
                header = next(reader)
            except StopIteration:
                return ["Row 1 must be the header row; file is empty."]
            if header != REQUIRED_HEADER:
                return [f"Row 1 (header) must be exactly: {','.join(REQUIRED_HEADER)}"]
            seen_ranks = set()
            by_rank = []
            for row_num, row in enumerate(reader, start=2):
                if not row: continue
                if len(row) < 4:
                    errors.append(f"Row {row_num}: Must have 4 elements.")
                    continue
                cid, rank_s, score_s = row[0], row[1], row[2]
                if not CANDIDATE_ID_PATTERN.match(cid):
                    errors.append(f"Row {row_num}: Invalid candidate_id '{cid}'.")
                try:
                    rank = int(rank_s)
                    seen_ranks.add(rank)
                except ValueError:
                    errors.append(f"Row {row_num}: rank must be an integer.")
                    rank = None
                try:
                    score = float(score_s)
                except ValueError:
                    errors.append(f"Row {row_num}: score must be a float.")
                    score = None
                if rank is not None and score is not None:
                    by_rank.append((rank, score, cid))
            missing = set(range(1, 101)) - seen_ranks
            if missing:
                errors.append(f"Missing ranks: {sorted(missing)}")
            by_rank.sort(key=lambda x: x[0])
            for i in range(len(by_rank) - 1):
                if by_rank[i][1] < by_rank[i+1][1]:
                    errors.append(f"Score violation: rank {by_rank[i][0]} < rank {by_rank[i+1][0]}.")
                if by_rank[i][1] == by_rank[i+1][1] and by_rank[i][2] > by_rank[i+1][2]:
                    errors.append(f"Tie-breaker error at ranks {by_rank[i][0]} and {by_rank[i+1][0]}.")
    except Exception as e:
        errors.append(f"Could not read file: {e}")
    return errors

if __name__ == '__main__':
    issues = validate_submission('submission.csv')
    if not issues:
        print("🎉 SUCCESS: Your submission file is flawless and perfectly formatted!")
    else:
        print("❌ VALIDATION FAILED:")
        for err in issues: print(f"  - {err}")
"""
    with open("validate_submission.py", "w", encoding="utf-8") as f:
        f.write(validate_code.strip())
    print("✅ validate_submission.py has been deployed to the workspace directory.")

In [ ]:
if __name__ == "__main__":
    if os.path.exists(TARGET_FILE):
        # 1. Execute stream cleaning filter (Step 1)
        clean_pool = stream_and_filter_candidates(TARGET_FILE)

        # 2. Compute normalizations and batched text vector structures (Steps 2-5)
        run_production_semantic_pipeline(clean_pool, output_filename=OUTPUT_FILE)

        # 3. Deploy local automated validator environment
        deploy_validation_script()
    else:
        print(f"⚠️ Missing critical '{TARGET_FILE}' source dataset in current workspace directory!")

🚀 Open and streaming file data pipeline: candidates.jsonl...

🏁 --- Streaming Phase Complete ---
Total Profiles Streamed: 100000
❌ Honeypot Traps Defused & Removed: 16388
❌ IT Service Giant Profiles Excluded: 0
✅ High-Quality Candidates Passed to Pool: 83612
📦 Loading local semantic embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Extracting narratives and computing technical skills for all 83612 candidates...
📦 Vectorizing profiles via Batched Sentence Transformers (MiniLM)...


Batches:   0%|          | 0/654 [00:00<?, ?it/s]

🎯 Calculating structural cosine similarities against Job Description...
⚖️ Applying Min-Max scaling to equalize score boundaries...
🔄 Blending indices and computing behavioral platform weights...
📝 Writing exactly 100 highly calibrated records to 'submission.csv'...
🏁 SUCCESS! Standalone semantic-first submission exported to 'submission.csv'
✅ validate_submission.py has been deployed to the workspace directory.
